In [1]:

# Install packages if needed
# !pip install folium geopy feedparser pandas numpy

import re
import time
from datetime import datetime

import pandas as pd
import numpy as np
import feedparser
import folium

from folium.plugins import MarkerCluster, HeatMap
from geopy.geocoders import Nominatim

# ===============================
# CONFIGURATION
# ===============================

RSS_FEEDS = [
    "https://feeds.reuters.com/reuters/worldNews",
    "https://www.aljazeera.com/xml/rss/all.xml",
    "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
    "https://www.theguardian.com/world/middleeast/rss",
    "https://www.bbc.com/news/world/middle_east/rss.xml"
]

MAP_CENTER = [26.0, 50.0]
MAP_ZOOM = 5

MAP_FILE = "conflict_map.html"
DASHBOARD_FILE = "conflict_dashboard.html"

PRIMARY_TERMS = [
    "iran","israel","gaza","lebanon","tehran","tel aviv","jerusalem",
    "saudi arabia","riyadh","uae","dubai","abu dhabi","qatar","doha",
    "kuwait","oman","hormuz","persian gulf","red sea","basra","beirut"
]

ACTION_TERMS = [
    "strike","missile","attack","drone","bomb","rocket","raid",
    "explosion","blast","refinery","pipeline","airport","port",
    "power plant","substation","military base","naval base"
]

PLACE_HINTS = {
    "Tehran":"Tehran, Iran",
    "Tel Aviv":"Tel Aviv, Israel",
    "Jerusalem":"Jerusalem, Israel",
    "Dubai":"Dubai, UAE",
    "Abu Dhabi":"Abu Dhabi, UAE",
    "Riyadh":"Riyadh, Saudi Arabia",
    "Doha":"Doha, Qatar",
    "Basra":"Basra, Iraq",
    "Beirut":"Beirut, Lebanon",
    "Hormuz":"Strait of Hormuz"
}

HIGH_SEVERITY = ["missile","strike","attack","explosion","bomb"]

geolocator = Nominatim(user_agent="conflict_map")
geo_cache = {}

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")
    return re.sub(r"\s+", " ", text).strip()

def geocode_location(loc):
    if loc in geo_cache:
        return geo_cache[loc]
    try:
        point = geolocator.geocode(loc)
        if point:
            geo_cache[loc] = {"lat":point.latitude,"lon":point.longitude}
            return geo_cache[loc]
    except:
        pass
    return None

def extract_location(text):
    t = text.lower()
    for hint,canon in PLACE_HINTS.items():
        if hint.lower() in t:
            return canon
    return None

def severity_score(text):
    score = 3
    t = text.lower()
    for w in HIGH_SEVERITY:
        if w in t:
            score += 4
    return min(score,15)

def alert_level(s):
    if s>=12: return "Critical"
    if s>=8: return "High"
    if s>=5: return "Medium"
    return "Low"

def parse_rss():
    incidents=[]
    for url in RSS_FEEDS:
        try:
            feed=feedparser.parse(url)
        except:
            continue

        for entry in feed.entries[:20]:
            title=clean_text(getattr(entry,"title",""))
            summary=clean_text(getattr(entry,"summary",""))
            text=f"{title} {summary}".lower()

            if not any(k in text for k in PRIMARY_TERMS) and not any(k in text for k in ACTION_TERMS):
                continue

            loc=extract_location(text)
            if not loc:
                continue

            geo=geocode_location(loc)
            time.sleep(1)

            if not geo:
                continue

            sev=severity_score(text)
            level=alert_level(sev)

            incidents.append({
                "time":datetime.utcnow().strftime("%Y-%m-%d %H:%M"),
                "title":title,
                "location":loc,
                "lat":geo["lat"],
                "lon":geo["lon"],
                "severity":sev,
                "level":level
            })

    return incidents

def build_map(data):

    m=folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles="CartoDB positron"
    )

    cluster=MarkerCluster().add_to(m)

    for i in data:

        color={
            "Critical":"darkred",
            "High":"red",
            "Medium":"orange",
            "Low":"green"
        }[i["level"]]

        folium.CircleMarker(
            location=[i["lat"],i["lon"]],
            radius=max(6,i["severity"]),
            color=color,
            fill=True,
            fill_opacity=0.7,
            popup=f"{i['title']}<br>{i['location']}"
        ).add_to(cluster)

    if data:
        HeatMap([[x["lat"],x["lon"],x["severity"]] for x in data]).add_to(m)

    return m

def build_dashboard(df):

    table=df.to_html(index=False)

    html = f"""
<html>
<head>
<title>Conflict Dashboard</title>
<style>
body {{font-family: Arial; margin:20px;}}
iframe {{width:100%; height:650px; border:1px solid #ccc;}}
table {{border-collapse:collapse;width:100%;}}
th,td {{border:1px solid #ddd;padding:6px;}}
th {{background:#eee;}}
</style>
</head>

<body>

<h1>Conflict Monitoring Dashboard</h1>

<iframe src='{MAP_FILE}'></iframe>

<h2>Events</h2>

{table}

</body>
</html>
"""

    with open(DASHBOARD_FILE,"w") as f:
        f.write(html)


In [2]:

incidents=parse_rss()

print("Incidents detected:",len(incidents))

df=pd.DataFrame(incidents)

display(df)

m=build_map(incidents)

m.save(MAP_FILE)

build_dashboard(df)

print("Files created:")
print(MAP_FILE)
print(DASHBOARD_FILE)

m


Incidents detected: 11


,time,title,location,lat,lon,severity,level
0,2026-03-07 22:30,Smoke rises from a Dubai tower after attack,"Dubai, UAE",25.074282,55.188539,7,Medium
1,2026-03-07 22:30,British family stranded in Middle East after F...,"Dubai, UAE",25.074282,55.188539,7,Medium
2,2026-03-07 22:30,'Don't die': the two words that sum up our liv...,"Tehran, Iran",35.689252,51.389600,15,Critical
3,2026-03-07 22:30,Tehran warns Europe to stay out of conflict or...,"Tehran, Iran",35.689252,51.389600,7,Medium
4,2026-03-07 22:30,US considers lifting more sanctions on Russian...,"Tehran, Iran",35.689252,51.389600,7,Medium
5,2026-03-07 22:30,Trump demands Iran’s ‘unconditional surrender’...,"Tehran, Iran",35.689252,51.389600,11,High
6,2026-03-07 22:30,Trump demands 'unconditional surrender' from I...,"Tehran, Iran",35.689252,51.389600,3,Low
7,2026-03-07 22:30,'We couldn't sleep because of fear': Residents...,"Beirut, Lebanon",33.889226,35.502558,3,Low
8,2026-03-07 22:30,Iran's high-risk war strategy seems to centre ...,"Tehran, Iran",35.689252,51.389600,7,Medium
9,2026-03-07 22:30,Israel strikes Beirut after evacuation warning...,"Beirut, Lebanon",33.889226,35.502558,7,Medium


Files created:
conflict_map.html
conflict_dashboard.html
